#### 주의!!

이 실습은 가급적 NVIDIA GPU가 설치된 컴퓨터 환경이거나 Google Colab에서 진행해주세요.

- OpenAI API 자료 레퍼런스: https://platform.openai.com/docs/guides/text-generation
- 실습 모델 레퍼런스: https://huggingface.co/LGAI-EXAONE
- 실습 데이터 레퍼런스
    - NSMC: https://huggingface.co/datasets/e9t/nsmc
    - KorQuAD: https://huggingface.co/datasets/KorQuAD/squad_kor_v1

## 환경 준비

# 프롬프트만으로 어디까지 되나

학습을 하지 않고 **말로만** 시켜서 어디까지 되는지 봅니다.

이것을 먼저 하는 데는 이유가 있습니다. 학습은 비쌉니다. 데이터를 모으고, 평가 체계를
만들고, GPU 를 돌리고, 그 다음에도 유지보수가 따라붙습니다.
**프롬프트로 되면 그걸 다 안 해도 됩니다.**

## 무엇을 하게 되나

1. **지시만으로** 감정을 분류해 봅니다 (zero-shot)
2. 출력이 제멋대로인 것을 보고 **형식을 잡아갑니다**
3. **예시를 몇 개 보여주고**(few-shot) 얼마나 나아지는지 잽니다
4. 문장에서 **정보를 뽑아내는** 일에도 같은 방법을 씁니다

## 이 실습의 결론은 숫자입니다

3번에서 **zero-shot 정확도와 few-shot 정확도를 비교**하게 됩니다.
그 두 숫자의 차이가 "예시를 보여주는 것이 값어치가 있는가" 에 대한 답입니다.

> 학습은 하지 않습니다. GPU 는 추론에만 씁니다.


In [1]:
# vLLM 은 별도 venv 에서 서버로 띄운다 (verify/02_vllm_venv.sh 참조).
# 이 노트북은 HTTP 로 붙기만 하므로 openai 클라이언트만 있으면 된다.
# openai 는 3.x 로 올리지 않는다. llama-index-llms-openai 가 openai<3 을
# 요구해서, 3일차 RAG 실습과 같은 환경을 쓰려면 2.x 여야 한다.
%pip install -q -U 'openai<3' datasets

여러분이 실습할 모델을 실행하는 명령어 입니다. 실행후 강사님의 지시가 있을때 까지 다른 코드들을 실행하지 마세요.

In [2]:
# 구 실행 방식(python -m vllm.entrypoints.openai.api_server)은 deprecated 다.
# 그리고 vLLM 은 torch 를 하드핀하므로 학습 환경과 같은 venv 에 깔면 안 된다.
# 아래는 별도 터미널에서 실행한다:
#   source /opt/vllm-env/bin/activate
#   nohup vllm serve Qwen/Qwen3-4B-Instruct-2507 --port 8000 > vllm.log 2>&1 &
# 기동 확인:
!curl -s http://localhost:8000/v1/models || echo "서버가 아직 준비되지 않았습니다"

nohup: appending output to 'nohup.out'


In [3]:
from openai import OpenAI
from datasets import load_dataset

### 두 가지 호출 방식

바로 아래 두 셀이 거의 같아 보이지만 **API 가 다릅니다.**

| | `completions` | `chat.completions` |
|---|---|---|
| 입력 | 문자열 하나 | **역할이 있는 메시지 목록** |
| 성격 | 이어 쓰기 | 주고받기 |

`chat` 쪽은 `system` / `user` / `assistant` 로 역할을 나눕니다.
`system` 에 "너는 무엇이다" 를 두고 `user` 에 실제 질문을 두는 것이 관례입니다.

요즘 모델은 대부분 `chat` 형식으로 학습돼 있어서, **`chat` 을 쓰는 것이 기본**입니다.
`completions` 는 이어 쓰기가 필요한 특수한 경우에만 씁니다.

3일차 SFT 실습에서 본 챗 템플릿이 바로 이 형식을 문자열로 바꾸는 장치였습니다.

In [4]:
# 모델이 잘 실행되는지 테스트 해봅니다.
openai_api_key = "EMPTY"
openai_api_base = "http://localhost:8000/v1"

client = OpenAI(
    api_key=openai_api_key,
    base_url=openai_api_base,
)
completion = client.completions.create(model="Qwen/Qwen3-4B-Instruct-2507",
                                       prompt="대한민국에 대해 소개해주세요.")
print("Completion result:", completion.choices[0].text)

Completion result: 
영향력과 부조리 + 13억orphism (MKILL) 이거


In [4]:
completion = client.chat.completions.create(
    model="Qwen/Qwen3-4B-Instruct-2507",
    messages=[
        {
            "role": "system",
            "content": "You are a helpful assistant."
        },
        {
            "role": "user",
            "content": "대한민국에 대해 소개해주세요."
        },
        {
            "role": "assistant",
            "content": "대한민국, 공식적으로는 한국으로도 알려진 이 나라는 아시아 대륙의 동북부에 위치합니다."
        },
        {
            "role": "user",
            "content": "왜 그럴까?"
        },
    ]
)

print(completion.choices[0].message.content)

대한민국이 아시아 대륙의 동북부에 위치하는 이유는 지리적 위치와 지명의 유래 때문입니다:

1. **지리적 위치**:
   - **경계**: 대한민국은 동쪽으로는 동해(일본해로도 알려져 있음), 서쪽으로는 서해(한반도 서쪽의 바다), 남쪽으로는 동해가 위치해 있으며, 북쪽으로는 중국과 북한이 인접해 있습니다.
   - **교 regurg산 산맥**: 이 산맥이 한국 본토를 둘러싸고 있어 동쪽과 서쪽으로 나뉘어진 지형적 특성을 가지고 있습니다.

2. **지명의 유래**:
   - **"언덕 동쪽 나라"**: 대한민국의 이름은 "한"나라(韓國)의 약자인 "韓"에서 유래되었습니다. "한"은 고대부터 한반도 지역을 가리키는 용어였고, "국"은 나라를 의미합니다. 따라서 "నలున్న" (원래는 "낮"이라는 뜻)와 "한"을 합성하여 "한국"이라 불리게 되었습니다. 역사적으로는seat되게 유래되었으며, "한"은 한반도의 역사적 지역을 가리키는 표현이었습니다.

이러한 지리적 특성과 역사적 배경이 정리되어 오늘날의 대한민국이 아시아 동북부에 위치하게 만드는 주요 요인입니다.


### 감정 분석

In [5]:
# e9t/nsmc 는 로딩 스크립트 방식이라 datasets 5.x 에서 읽지 못한다.
dataset = load_dataset("e9t/nsmc", revision="refs/convert/parquet")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/3.74k [00:00<?, ?B/s]

nsmc.py:   0%|          | 0.00/3.18k [00:00<?, ?B/s]

In [6]:
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'document', 'label'],
        num_rows: 150000
    })
    test: Dataset({
        features: ['id', 'document', 'label'],
        num_rows: 50000
    })
})

In [7]:
dataset['train'][0]

{'id': '9976970', 'document': '아 더빙.. 진짜 짜증나네요 목소리', 'label': 0}

In [8]:
df = dataset['train'].to_pandas()

In [9]:
df.head(5)

,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...,1


### 간단한 감정분석

In [11]:
SYSTEM_PROMPT = "You are a helpful assistant."
TASK_PROMPT = "주어진 문장에 대해 감정 분석을 해주세요. 감정 분석은 '긍정', '부정' 으로 표현해주세요."

In [12]:
messages=[
        {
            "role": "system", "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": TASK_PROMPT + "\n\n" + dataset['test'][10]['document']
        }
    ]

completion = client.chat.completions.create(
    model="Qwen/Qwen3-4B-Instruct-2507",
    messages=messages
)

print('분석 문장: ', dataset['test'][10]['document'])
print(completion.choices[0].message.content)

분석 문장:  괜찮네요오랜만포켓몬스터잼밌어요
주어진 문장 **"괜찮네요오랜만포켓몬스터잼슐리요"** 에 대한 감정 분석 결과는 다음과 같습니다:

- **주요 감정**: **긍정**
  - **설명**: 문장에서 "잼 SDI요"는 즐거운 반응을 나타내며, "오랜만포켓몬스터" 부분은 기대감과 긍정적인 추억을 연상시킵니다. "괜찮네요"라는 표현은 상황이 대체로 수용 가능하다는 긍정적인 태도를 보여줍니다. 따라서 전체적으로 긍정적인 감정이 주된 경향을 이루고 있습니다.


In [13]:
for i in range(10):
    messages=[
            {
                "role": "system", "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": TASK_PROMPT + "\n\n" + dataset['test'][i]['document']
            }
        ]

    completion = client.chat.completions.create(
        model="Qwen/Qwen3-4B-Instruct-2507",
        messages=messages
    )
    print("- 분석 텍스트: ", dataset['test'][i]['document'])
    print("- 분석 결과: ", completion.choices[0].message.content)
    print('*' * 10)

- 분석 텍스트:  굳 ㅋ
- 분석 결과:  주어진 문장 "굳 ᄏ"은 한국에서 자주 사용되는 비표준 표현으로, 문맥에 따라 다양한 해석이 가능하지만 주로 긍정적인 의미로 해석될 가능성이 높습니다. 여기서는 간결하게 분석해보겠습니다:

- **감정 분석 결과**: **긍정**

문장이 짧고 특정 상황에 따라 긍정적인반응이나 확신을 나타낼 수 있기 때문에 주로 긍정적인 감정으로 분류할 수 있습니다. 하지만 문맥에 따라 부정적 의미로도 해석될 수 있다는 점을 고려하셔야 합니다. 더 구체적인 의미 파악을 위해서는 주변 문맥이 필요할 수 있습니다.
**********
- 분석 텍스트:  GDNTOPCLASSINTHECLUB
- 분석 결과:  주어진 텍스트 "GDNTOPCLASSINTHECLUB"는 명확한 의미나 문맥을 파악하기 어렵습니다. 이는 아마도 특수 문자, 코드, 또는 줄임말이 섞여 있는 것 같습니다. 감정 분석을 위해서는 더 자세한 문맥이나 설명이 필요합니다. 감정 분석을 정확히 Categories ('긍정', '부정')로 제한하려면, 텍스트가 어떤 감정을 전달하려는지에 대한 추가 정보가 필수적입니다.

만약 특정 설정이나 의도를 가진 텍스트라고 가정한다면, 현재로선 분석이 불가능합니다. 더 정확한 분석을 위해서는 생성자님의 의도가 무엇인지 알려주시면 감사하겠습니다.
**********
- 분석 텍스트:  뭐야 이 평점들은.... 나쁘진 않지만 10점 짜리는 더더욱 아니잖아
- 분석 결과:  이 문장의 감정 분석 결과는 **부정**입니다. 

문제점은 "나쁘진 않지만"이라는 표현에서 계층적인 부정적인 뉘앙스를 보여주고 있으며, 특히 "10점 짜리는 더더욱 아니잖아"라는 부분에서 기대치에 미치지 못하는 실망이나 불만족이 명확히 드러납니다. 따라서 전체적으로 긍정적인 관점보다는 부정적인 감정이 predominance한다고 볼 수 있습니다.
**********
- 분석 텍스트:  지루하지는 않은데 완전 막장임... 돈주고 보기에는....
- 분석 결과:  이 문장에

### 출력이 제멋대로인 것을 보세요

바로 위 셀에서 10건을 돌렸습니다. 결과가 지저분할 겁니다.

- 어떤 건 `긍정`, 어떤 건 `이 리뷰는 긍정적입니다`
- 어떤 건 이유까지 설명
- 어떤 건 앞에 인사말

**모델이 틀린 게 아닙니다.** 우리가 형식을 말해주지 않았을 뿐입니다.

이 상태로는 프로그램에서 쓸 수 없습니다. 결과를 세려면 `긍정`인지 아닌지
**기계가 판별**할 수 있어야 하니까요.

아래에서 형식을 잡아갑니다. 세 단계로 점점 세게 조입니다.

| 방법 | 강제력 |
|---|---|
| 프롬프트로 **부탁**한다 | 약함 — 대체로 따르지만 보장 없음 |
| `seed` 로 **재현성**을 확보한다 | 흔들림만 줄임 |
| **제약 디코딩**으로 막는다 | **강함 — 다른 답이 나올 수 없음** |

In [14]:
SYSTEM_PROMPT = "You are a helpful assistant."
TASK_PROMPT = """주어진 문장에 대해 감정 분석을 해주세요. 감정 분석은 '긍정', '부정' 으로 표현해주세요.
분석 결과 출력 시 다음의 형식에 맞게 작성해주세요.

- 분석 결과: (긍정 or 부정)
"""

In [15]:
messages=[
        {
            "role": "system", "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": TASK_PROMPT + "\n\n" + dataset['test'][0]['document']
        }
    ]

completion = client.chat.completions.create(
    model="Qwen/Qwen3-4B-Instruct-2507",
    messages=messages,
    seed=1234,
)

print(completion.choices[0].message.content)

- 분석 결과: 긍정


In [21]:
completion = client.chat.completions.create(
    model="Qwen/Qwen3-4B-Instruct-2507",
    messages=messages,
    seed=7,
)

print(completion.choices[0].message.content)

분석 결과: **긍정** 

이 분석은 문장이 짧고 특정 단어 선택이 긍정적인 톤을 시사하기 때문입니다. '굳튬'과 '커널' 같은 단어들은 대체로 안정감과 견고함을 연상시키는 경향이 있어 긍정적인 맥락에서 사용될 수 있습니다. 하지만 문맥에 따라 해석이 달라질 수 있음을 참고하시길 바랍니다.


In [22]:
completion = client.chat.completions.create(
    model="Qwen/Qwen3-4B-Instruct-2507",
    messages=messages,
    n=10
)

for c in completion.choices:
    print(c.message.content)
    print('*' * 30)

- 분석 결과: 긍정
******************************
Analysis Result: 부정
******************************
- 분석 결과: 긍정
******************************
- 분석 결과: 긍정
******************************
분석 결과: 부정
******************************
- 분석 결과: 긍정
******************************
분석 결과: 부정
******************************
분석 결과: 부정
******************************
- 분석 결과: 부정
******************************
감정 분석 결과: **긍정** 

**설명:** "굳 탄다"는 일반적으로 강한 의지, 결단력, 굳센 성격 등 긍정적인 의미로 해석될 수 있습니다. 하지만 문맥에 따라 다소 모호할 수 있으므로, 주로 긍정적인 뉘앙스를 띠는 경향이 큽니다.
******************************


### 여기부터는 같은 입력에 파라미터만 바꿉니다

아래 몇 셀은 **같은 질문을 반복**합니다. 바뀌는 것은 파라미터뿐입니다.

| 파라미터 | 하는 일 |
|---|---|
| `seed` | 난수를 고정한다. 같은 seed 면 같은 답 |
| `n` | 한 번에 여러 개를 뽑는다. **얼마나 흔들리는지** 보는 용도 |
| `temperature` | 낮으면 보수적, 높으면 과감 |
| `top_p` | 확률 상위 몇 %에서만 고른다 |

**보려는 것은 답 자체가 아니라 답이 흔들리는 폭입니다.**

같은 입력인데 답이 매번 다르다면, 그 프롬프트로 만든 결과는 신뢰하기 어렵습니다.
2일차 프롬프트 최적화에서 "judge 점수에 노이즈가 있다" 고 했던 것도 같은 이야기입니다.

분류처럼 답이 정해진 일에는 `temperature` 를 낮게 둡니다.

In [28]:
completion = client.chat.completions.create(
    model="Qwen/Qwen3-4B-Instruct-2507",
    messages=messages,
    n=10,
    top_p=0.95,
    temperature=0.9,
)

for c in completion.choices:
    print(c.message.content)
    print('*' * 30)

- 분석 결과: 긍정
******************************
- 분석 결과: 긍정
******************************
분석 결과: 부정
******************************
- 분석 결과: 부정
******************************
분석 결과: 긍정
******************************
분석 결과: 부정
******************************
- 분석 결과: 부정
******************************
- 분석 결과: 긍정
******************************
- 분석 결과: 긍정
******************************
- 분석 결과: 긍정
******************************


In [29]:
completion = client.chat.completions.create(
    model="Qwen/Qwen3-4B-Instruct-2507",
    messages=messages,
    n=10,
    top_p=0.5,
    temperature=0.5,
)

for c in completion.choices:
    print(c.message.content)
    print('*' * 30)

- 분석 결과: 긍정
******************************
- 분석 결과: 긍정
******************************
- 분석 결과: 긍정
******************************
- 분석 결과: 긍정
******************************
- 분석 결과: 긍정
******************************
- 분석 결과: 긍정
******************************
- 분석 결과: 긍정
******************************
- 분석 결과: 긍정
******************************
- 분석 결과: 긍정
******************************
- 분석 결과: 긍정
******************************


### 제약 디코딩 — 다른 답이 나올 수 없게 막는다

지금까지는 프롬프트로 **부탁**했습니다. 아래는 다릅니다.

```python
extra_body={"structured_outputs": {"choice": ["긍정", "부정"]}}
```

이렇게 주면 모델은 **둘 중 하나밖에 출력할 수 없습니다.**
부탁이 아니라 **문법으로 막는** 것입니다. 생성 단계에서 다른 토큰의 확률을 0으로 만듭니다.

프롬프트를 아무리 잘 써도 가끔 형식을 어깁니다. 1,000건 중 3건이면 그 3건이
파이프라인을 멈춥니다. 제약 디코딩은 그 가능성을 없앱니다.

**아래 평가 루프부터는 전부 이 방식을 씁니다.** 그래야 세는 것이 가능합니다.

> 뒤쪽 QA 절에서는 같은 원리를 **JSON 스키마**로 씁니다.
> 선택지가 아니라 구조를 강제하는 형태입니다.

In [30]:
completion = client.chat.completions.create(
    model="Qwen/Qwen3-4B-Instruct-2507",
    messages=messages,
    extra_body={"structured_outputs": {"choice": ["긍정", "부정"]}},
    seed=0,
    top_p=0.9,
    temperature=0.8,
)
print(completion.choices[0].message.content)

부정


## 재보기 — 여기서부터 숫자가 나옵니다

10건을 돌려 정확도를 잽니다. 흐름은 이렇습니다.

```
10건 반복 호출 → 긍정/부정 수집 → 1/0 으로 변환 → 정답과 비교 → 정확도
```

> **10건은 통계적으로 의미가 없습니다.** 한 건만 달라져도 10%가 움직입니다.
> 실습이라 작게 잡은 것이고, 실제로는 최소 수백 건을 씁니다.
> 그래도 **비교의 방향**은 볼 수 있습니다.

이 숫자를 기억해 두세요. 아래 few-shot 절에서 **같은 루프를 다시 돌려**
두 숫자를 비교하는 것이 이 노트북의 결론입니다.

In [31]:
preds = []

for i in range(10):
    messages=[
            {
                "role": "system", "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": TASK_PROMPT + "\n\n" + dataset['test'][i]['document']
            }
        ]

    completion = client.chat.completions.create(
        model="Qwen/Qwen3-4B-Instruct-2507",
        messages=messages,
        extra_body={"structured_outputs": {"choice": ["긍정", "부정"]}},
        seed=1234,
        top_p=0.9,
        temperature=0.8,
    )

    preds.append(completion.choices[0].message.content)

In [32]:
preds

['부정', '부정', '부정', '부정', '부정', '긍정', '부정', '부정', '부정', '부정']

In [33]:
pred_labels = []

for p in preds:
    if p == '긍정':
        pred_labels.append(1)
    else:
        pred_labels.append(0)

In [34]:
pred_labels

[0, 0, 0, 0, 0, 1, 0, 0, 0, 0]

In [37]:
ground_truth = dataset['test']['label'][:10]

In [38]:
# 정확도는 얼마나 되는지 구현해주세요.
count = 0

for i in range(10):
  if pred_labels[i] == ground_truth[i]:
    count += 1

print(count/len(ground_truth))

0.8


### 같은 루프를 few-shot 으로 다시 돌립니다

아래는 위 평가 루프와 **거의 같은 코드**입니다. 프롬프트만 바뀝니다.

```
zero-shot :  지시            → 분류
few-shot  :  지시 + 예시 5개  → 분류
```

예시를 `train[100:105]` 에서 그냥 잘라 씁니다. **라벨이 균형 잡혀 있는지 확인하지 않습니다.**
5개가 전부 긍정이면 모델이 긍정 쪽으로 기웁니다 — 실무에서는 이걸 확인합니다.

**두 정확도를 비교하세요.** 그것이 이 노트북의 결론입니다.

| 결과 | 뜻 |
|---|---|
| few-shot 이 높다 | 예시가 도움이 됐다 |
| 비슷하다 | 이 작업에는 지시만으로 충분하다 |
| few-shot 이 낮다 | 예시가 편향됐거나 문제를 헷갈리게 했다 |

**어느 쪽이든 배울 것이 있습니다.** few-shot 이 항상 낫지는 않습니다.

In [39]:
SYSTEM_PROMPT = "You are a helpful assistant."
TASK_PROMPT = """주어진 문장에 대해 감정 분석을 해주세요. 감정 분석은 '긍정', '부정' 으로 표현해주세요.
분석은 다음의 예제를 따라주세요.
"""

In [57]:
dataset['train'][100:105]

{'id': ['10044377', '6158844', '1723799', '10273782', '1666795'],
 'document': ['신카이 마코토의 작화와,미유와 하나카나가 연기를 잘해줘서 더대박이였다.',
  '재미없음 진심 1이훨나 캐스팅두못한듯',
  '잔잔한게 생각보다 볼만한 영화인거 같습니다 ㅋ',
  '감독님들 고은님 쓰면 영화안봅니다 .',
  '무섭지도 않았고 스토리도 ..ㅡㅡ'],
 'label': [1, 0, 1, 0, 0]}

In [58]:
example_text = []

documents = []
labels = []

for row in dataset['train']['document'][100:105]:
    documents.append(row)

for row in dataset['train']['label'][100:105]:
    if row == 0:
        labels.append('부정')
    else:
        labels.append('긍정')

In [59]:
for d, l in zip(documents, labels):
    example_text.append('문장: ' + d + '\n예측: ' + l)

In [60]:
example_text

['문장: 신카이 마코토의 작화와,미유와 하나카나가 연기를 잘해줘서 더대박이였다.\n예측: 긍정',
 '문장: 재미없음 진심 1이훨나 캐스팅두못한듯\n예측: 부정',
 '문장: 잔잔한게 생각보다 볼만한 영화인거 같습니다 ㅋ\n예측: 긍정',
 '문장: 감독님들 고은님 쓰면 영화안봅니다 .\n예측: 부정',
 '문장: 무섭지도 않았고 스토리도 ..ㅡㅡ\n예측: 부정']

In [61]:
FEW_SHOT_PROMPT = TASK_PROMPT + '\n\n'.join(example_text)

In [62]:
preds = []

for i in range(10):
    messages=[
            {
                "role": "system", "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": FEW_SHOT_PROMPT + "\n\n" + dataset['test'][i]['document']
            }
        ]

    completion = client.chat.completions.create(
        model="Qwen/Qwen3-4B-Instruct-2507",
        messages=messages,
        extra_body={"structured_outputs": {"choice": ["긍정", "부정"]}},
        seed=0,
        top_p=0.9,
        temperature=0.8,
    )

    preds.append(completion.choices[0].message.content)

In [63]:
preds

['긍정', '긍정', '긍정', '긍정', '부정', '긍정', '긍정', '부정', '부정', '부정']

In [64]:
pred_labels = []

for p in preds:
    if p == '긍정':
        pred_labels.append(1)
    else:
        pred_labels.append(0)

In [56]:
match_count = 0

for p, l in zip(pred_labels, dataset['test']['label'][:10]):
    if p == l:
      match_count += 1

match_count / 10

0.5

### 정보 추출 — 형식을 어떻게 강제하나

앞에서는 **분류**였습니다. 답이 둘 중 하나라 `choice` 로 막을 수 있었습니다.

여기서는 **뽑아내기**입니다. 지문에서 인물 이름을 찾아 목록으로 만듭니다.
답이 몇 개일지 모르고 내용도 미리 알 수 없으니 `choice` 를 쓸 수 없습니다.

세 단계로 진행합니다. **차이를 보는 것이 요점입니다.**

| | 방법 | 결과 |
|---|---|---|
| 1 | 그냥 물어본다 | 자유 문장. 파싱 불가 |
| 2 | 프롬프트에 JSON 예시를 넣는다 | 대체로 JSON. **보장은 없음** |
| 3 | **스키마로 강제**한다 | 항상 그 구조 |

2번과 3번의 차이가 핵심입니다. 2번은 잘 되는 것처럼 보이다가 가끔 깨집니다.
그 "가끔" 이 운영에서 사고가 됩니다.

3번은 pydantic 모델을 JSON 스키마로 바꿔 `response_format` 에 넘깁니다.
2일차 Amazon 실습에서 쓴 것과 같은 방법입니다.

> **스키마에 길이 상한을 두는 것**을 잊지 마세요. 문자열 필드에 제한이 없으면
> 모델이 끝없이 쓰다가 `max_tokens` 에서 잘리고, JSON 이 깨집니다.
> 2일차에 실제로 그것 때문에 파이프라인이 죽은 적이 있습니다.

In [65]:
qa_dataset = load_dataset("KorQuAD/squad_kor_v1")

README.md:   0%|          | 0.00/6.29k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/11.6M [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/60407 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5774 [00:00<?, ? examples/s]

In [66]:
qa_dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 60407
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 5774
    })
})

In [67]:
qa_dataset['train'][0]

{'id': '6566495-0-0',
 'title': '파우스트_서곡',
 'context': '1839년 바그너는 괴테의 파우스트을 처음 읽고 그 내용에 마음이 끌려 이를 소재로 해서 하나의 교향곡을 쓰려는 뜻을 갖는다. 이 시기 바그너는 1838년에 빛 독촉으로 산전수전을 다 걲은 상황이라 좌절과 실망에 가득했으며 메피스토펠레스를 만나는 파우스트의 심경에 공감했다고 한다. 또한 파리에서 아브네크의 지휘로 파리 음악원 관현악단이 연주하는 베토벤의 교향곡 9번을 듣고 깊은 감명을 받았는데, 이것이 이듬해 1월에 파우스트의 서곡으로 쓰여진 이 작품에 조금이라도 영향을 끼쳤으리라는 것은 의심할 여지가 없다. 여기의 라단조 조성의 경우에도 그의 전기에 적혀 있는 것처럼 단순한 정신적 피로나 실의가 반영된 것이 아니라 베토벤의 합창교향곡 조성의 영향을 받은 것을 볼 수 있다. 그렇게 교향곡 작곡을 1839년부터 40년에 걸쳐 파리에서 착수했으나 1악장을 쓴 뒤에 중단했다. 또한 작품의 완성과 동시에 그는 이 서곡(1악장)을 파리 음악원의 연주회에서 연주할 파트보까지 준비하였으나, 실제로는 이루어지지는 않았다. 결국 초연은 4년 반이 지난 후에 드레스덴에서 연주되었고 재연도 이루어졌지만, 이후에 그대로 방치되고 말았다. 그 사이에 그는 리엔치와 방황하는 네덜란드인을 완성하고 탄호이저에도 착수하는 등 분주한 시간을 보냈는데, 그런 바쁜 생활이 이 곡을 잊게 한 것이 아닌가 하는 의견도 있다.',
 'question': '바그너는 괴테의 파우스트를 읽고 무엇을 쓰고자 했는가?',
 'answers': {'text': ['교향곡'], 'answer_start': [54]}}

In [68]:
qa_dataset['validation'][30]['question']

'헤이그가 군에서 퇴역한 년도는 몇년도입니까?'

In [69]:
qa_dataset['validation'][30]['context']

'헤이그는 닉슨 대통령이 그를 사성 장군과 육군 부참모로 진급시킬 때 집중 광선과 논쟁으로 들어갔다. 헤이그를 군사의 최상으로 밀어넣은 닉슨의 행동은 대통령의 남자들을 다양한 연방 대리법에서 권한의 직우들로 놓은 노력과 함께 일치였다. 하지만 그는 곧 백악관으로 돌아가 1973년부터 1974년까지 대통령 특별 보좌관을 지냈다. 워터게이트 사건이 일어난지 한달 후, 헤이그는 포위된 닉슨 대통령을 위한 치명적 역할을 하였다. 그일은 8월 닉슨의 사임과 제럴드 포드의 대통령으로 계승으로 이끈 협상들에서 헤이그가 수단이었던 우연이 아니었다. 곧 후에 헤이그는 미국 유럽 연합군 최고사령부의 최고 사령관으로 임명되었다. 그는 나토에서 다음 5년을 보내고 1979년 군에서 퇴역하여 미국 기술 주식 회사의 우두머리가 되었다.'

In [70]:
qa_dataset['validation'][30]['answers']

{'text': ['1979년'], 'answer_start': [363]}

In [71]:
messages=[
        {
            "role": "system", "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": qa_dataset['validation'][30]['question'] + "\n\n" + qa_dataset['validation'][30]['context']
        }
    ]

completion = client.chat.completions.create(
    model="Qwen/Qwen3-4B-Instruct-2507",
    messages=messages,
    seed=0,
)

print('- 분석 질의: ', qa_dataset['validation'][30]['question'] + "\n\n" + qa_dataset['validation'][30]['context'])
print('')
print('- 분석 결과: ', completion.choices[0].message.content)

- 분석 질의:  헤이그가 군에서 퇴역한 년도는 몇년도입니까?

헤이그는 닉슨 대통령이 그를 사성 장군과 육군 부참모로 진급시킬 때 집중 광선과 논쟁으로 들어갔다. 헤이그를 군사의 최상으로 밀어넣은 닉슨의 행동은 대통령의 남자들을 다양한 연방 대리법에서 권한의 직우들로 놓은 노력과 함께 일치였다. 하지만 그는 곧 백악관으로 돌아가 1973년부터 1974년까지 대통령 특별 보좌관을 지냈다. 워터게이트 사건이 일어난지 한달 후, 헤이그는 포위된 닉슨 대통령을 위한 치명적 역할을 하였다. 그일은 8월 닉슨의 사임과 제럴드 포드의 대통령으로 계승으로 이끈 협상들에서 헤이그가 수단이었던 우연이 아니었다. 곧 후에 헤이그는 미국 유럽 연합군 최고사령부의 최고 사령관으로 임명되었다. 그는 나토에서 다음 5년을 보내고 1979년 군에서 퇴역하여 미국 기술 주식 회사의 우두머리가 되었다.

- 분석 결과:  제시된 텍스트에 따르면, 헤이그 장군이 군에서 퇴역한 정확한 연도는 **1979년**입니다.


In [72]:
SYSTEM_PROMPT = "You are a helpful assistant."
TASK_PROMPT = """주어진 내용에서 인물 이름에 대해서 나열해주세요."""

In [73]:
messages=[
        {
            "role": "system", "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": TASK_PROMPT + "\n\n" + qa_dataset['validation'][30]['context']
        }
    ]

completion = client.chat.completions.create(
    model="Qwen/Qwen3-4B-Instruct-2507",
    messages=messages,
)

print('- 분석 결과: ', completion.choices[0].message.content)

- 분석 결과:  주어진 내용에서 언급된 인물 이름은 다음과 같습니다:

1. 닉슨 (Nikkson) - 미국의 대통령으로 언급됨.
2. 헤이그 (Hayg) - 주요 인물로 언급되며 닉슨 대통령의 군사 관련 결정에 중추적인 역할을 했음이 설명됨.
3. 제럴드 포드 ( Gerald Ford) - 닉슨의 후임 대통령으로 언급됨.
4. 나토 (NATO) - 국가 의지 공동체로, 직접 인물이 아닌 그룹이지만 문맥상 중요한 역할을 함.

이름이 정확하게 표기되지 않거나 특정 인물에 대한 명확한 최초äsnaming 표시가 주어진 텍스트 내에서 모호한 부분이 있어 정확히 식별되지 않는 경우도 있음을 양해 부탁드립니다. 주어진 정보만으로는 주요 인물들을 나열하였습니다. 만약 특정 인물에 대한 확인이 필요하다면 추가 정보나 문맥이 필요할 수 있습니다.


In [74]:
SYSTEM_PROMPT = "You are a helpful assistant."
TASK_PROMPT = """주어진 내용에서 인물 이름에 대해서 나열해주세요.
나열한 결과는 다음과 같이 작성해주세요.

{
  "names": ["name1", "name2", ...]
}"""

In [75]:
messages=[
        {
            "role": "system", "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": TASK_PROMPT + "\n\n" + qa_dataset['validation'][30]['context']
        }
    ]

completion = client.chat.completions.create(
    model="Qwen/Qwen3-4B-Instruct-2507",
    messages=messages,
)

print(completion.choices[0].message.content)

주어진 텍스트에서 명확히 언급된 인물 이름은 다음과 같습니다:

{
  "names": ["닉슨", "헤이그", "제럴드 포드"]
}


In [76]:
import json
from pydantic import BaseModel

class NameDescription(BaseModel):
    names: list[str]

json_schema = NameDescription.model_json_schema()

completion = client.chat.completions.create(
    model="Qwen/Qwen3-4B-Instruct-2507",
    messages=messages,
    response_format={"type": "json_schema",
                     "json_schema": {"name": "answer", "schema": json_schema}},
)

print(completion.choices[0].message.content)
name_list = json.loads(completion.choices[0].message.content)

{
  "names": ["헤이그", "닉슨", "제럴드 포드"]
}


In [77]:
json_schema

{'properties': {'names': {'items': {'type': 'string'},
   'title': 'Names',
   'type': 'array'}},
 'required': ['names'],
 'title': 'NameDescription',
 'type': 'object'}

In [ ]:
print(name_list, type(name_list))
print(name_list['names'])

{'names': ['닉슨', '헤이그']} <class 'dict'>
['닉슨', '헤이그']


In [ ]:
SYSTEM_PROMPT = "You are a helpful assistant."
TASK_PROMPT = """주어진 내용에서 연도 정보를 나열해주세요.
나열한 결과는 다음과 같이 작성해주세요.

1. 년도는 숫자만 표현할것
2. 숫자는 4자리로 반드시 표기할것
3. 텍스트 내용에만 있는 정보를 추출할 것

{
  "years": ["year1", "year2", ...]
}"""

In [ ]:
import json
from pydantic import BaseModel

class YearDescription(BaseModel):
    year: list[str]

json_schema = YearDescription.model_json_schema()

messages=[
        {
            "role": "system", "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": TASK_PROMPT + "\n\n" + qa_dataset['validation'][30]['context']
        }
    ]

completion = client.chat.completions.create(
    model="Qwen/Qwen3-4B-Instruct-2507",
    messages=messages,
    response_format={"type": "json_schema",
                     "json_schema": {"name": "answer", "schema": json_schema}},
)

print(completion.choices[0].message.content)
year_list = json.loads(completion.choices[0].message.content)

{"year": ["1973", "1974", "1979"]}


## 마무리

학습 없이 프롬프트만으로 어디까지 되는지 봤습니다.

- **형식을 말해주지 않으면** 모델은 제멋대로 답합니다. 모델 탓이 아닙니다
- 프롬프트로 **부탁**하는 것과 **제약 디코딩으로 막는** 것은 다릅니다.
  운영에서는 후자가 필요합니다
- **예시를 보여주는 것**(few-shot)이 항상 낫지는 않습니다. 재봐야 압니다
- 같은 입력에 답이 흔들리는 폭을 보세요. 그게 신뢰도입니다

### 그래서 학습은 언제 하나

여기까지가 **학습하지 않고 할 수 있는 것**입니다.
2일차에는 이것을 자동화하는 방법(프롬프트 최적화)도 봤습니다.

그래도 부족할 때 학습으로 갑니다. 다음 실습부터가 그것입니다 —
**SFT · DPO · GRPO**. 각각 필요한 데이터가 다르고, 비용도 다릅니다.

프롬프트로 얻은 이 숫자를 기억해 두세요. **학습이 그 위에 얼마를 더 얹어주는지**
가 판단 기준이 됩니다.